# Jerk Modes Demo

This notebook compares `jaccpot` jerk modes:
- `fast_approx`
- `accurate`

It also computes a direct-sum jerk reference for a small system.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from jaccpot import FastMultipoleMethod

In [ ]:
def sample_problem(n=128, seed=0, dtype=jnp.float32):
    key = jax.random.PRNGKey(seed)
    kpos, kmass, kvel = jax.random.split(key, 3)
    positions = jax.random.uniform(kpos, (n, 3), minval=-1.0, maxval=1.0, dtype=dtype)
    masses = jax.random.uniform(kmass, (n,), minval=0.5, maxval=1.5, dtype=dtype)
    velocities = jax.random.uniform(kvel, (n, 3), minval=-0.2, maxval=0.2, dtype=dtype)
    return positions, masses, velocities

def direct_sum_jerk(positions, masses, velocities, G=1.0, softening=1e-3):
    diff = positions[:, None, :] - positions[None, :, :]
    vdiff = velocities[:, None, :] - velocities[None, :, :]
    dist_sq = jnp.sum(diff * diff, axis=-1) + softening**2
    eps = jnp.finfo(positions.dtype).eps
    inv_r = jnp.where(dist_sq > 0, 1.0 / (jnp.sqrt(dist_sq) + eps), 0.0)
    inv_r3 = inv_r / dist_sq
    inv_r5 = inv_r3 / dist_sq
    eye = jnp.eye(positions.shape[0], dtype=bool)
    inv_r3 = jnp.where(eye, 0.0, inv_r3)
    inv_r5 = jnp.where(eye, 0.0, inv_r5)
    rv = jnp.sum(diff * vdiff, axis=-1)
    term = vdiff * inv_r3[..., None] - 3.0 * rv[..., None] * diff * inv_r5[..., None]
    weighted = masses[None, :, None] * term
    return -G * jnp.sum(weighted, axis=1)

In [ ]:
positions, masses, velocities = sample_problem(n=96, seed=7)
solver = FastMultipoleMethod(preset="balanced", basis="solidfmm")

acc_fast, jerk_fast = solver.compute_accelerations_and_jerk(
    positions, masses, velocities,
    leaf_size=16, max_order=4, jerk_mode="fast_approx"
)

acc_acc, jerk_acc = solver.compute_accelerations_and_jerk(
    positions, masses, velocities,
    leaf_size=16, max_order=4, jerk_mode="accurate", jerk_fd_dt=1e-3
)

print(acc_fast.shape, jerk_fast.shape)

In [ ]:
# Compare with direct sum on a smaller subset for reference cost.
n_ref = 48
pos_ref = positions[:n_ref]
mass_ref = masses[:n_ref]
vel_ref = velocities[:n_ref]

solver_ref = FastMultipoleMethod(preset="accurate", basis="solidfmm", theta=1e-4)
_, jerk_fast_ref = solver_ref.compute_accelerations_and_jerk(
    pos_ref, mass_ref, vel_ref, leaf_size=12, max_order=4, jerk_mode="fast_approx"
)
_, jerk_acc_ref = solver_ref.compute_accelerations_and_jerk(
    pos_ref, mass_ref, vel_ref, leaf_size=12, max_order=4, jerk_mode="accurate", jerk_fd_dt=1e-3
)
jerk_direct = direct_sum_jerk(pos_ref, mass_ref, vel_ref)

def rel_err(a, b):
    return float(jnp.linalg.norm(a - b) / (jnp.linalg.norm(b) + 1e-12))

print("rel_err fast_approx vs direct:", rel_err(jerk_fast_ref, jerk_direct))
print("rel_err accurate    vs direct:", rel_err(jerk_acc_ref, jerk_direct))

## Notes
- `fast_approx` is typically faster.
- `accurate` includes source-motion effects via finite differences and is typically slower.
- For production settings, benchmark both modes on your target problem sizes.